# Training Setup — RunPod
Works on **RunPod** (primary target) and **local Windows/Linux**. Run cells top-to-bottom. Setup cells (1–8) only need to run once per pod.

**RunPod tips**
- Mount a **persistent network volume** at `/workspace` so checkpoints and dataset survive pod restarts.
- Recommended pod: PyTorch 2.1 / CUDA 11.8 image, A4000/A5000/A6000 or 4090.
- The training cell streams output live and saves checkpoints to `models/last.pt` + `models/best.pt`. If the pod dies, restart and re-run cells 0–10, then set `RESUME = "models/last.pt"` in cell 10 to continue.

## 0. Detect platform

In [ ]:
import platform, sys, os

IS_WINDOWS = platform.system() == "Windows"
IS_RUNPOD  = os.path.exists("/workspace")

REPO_NAME = "image_captioning_with_vmamba"
REPO_URL  = "https://github.com/RATSAPORN/image_captioning_with_vmamba.git"  # change if your fork lives elsewhere

if IS_RUNPOD:
    PROJECT = f"/workspace/{REPO_NAME}"
else:
    PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(f"Platform : {platform.system()}")
print(f"RunPod   : {IS_RUNPOD}")
print(f"Project  : {PROJECT}")

## 1. Clone or pull latest code (RunPod only)
On a fresh pod, this clones the repo into `/workspace`. On subsequent runs it just pulls the latest commits.

In [ ]:
import subprocess
if IS_RUNPOD:
    if not os.path.exists(PROJECT):
        print(f"Cloning {REPO_URL} into {PROJECT}")
        subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)
    else:
        print(f"Pulling latest in {PROJECT}")
        subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
else:
    print("Local machine — skipping git pull. Make sure your code is up to date.")

## 2. Java for METEOR/CIDEr metrics
- **RunPod/Linux**: installs automatically
- **Windows**: install manually from https://adoptium.net (Temurin JRE 17)

In [ ]:
import shutil
if IS_WINDOWS:
    if shutil.which("java"):
        print("Java found:", shutil.which("java"))
    else:
        print("Java not found. Install from https://adoptium.net then restart the notebook.")
else:
    os.system("apt-get update -q && apt-get install -y --no-install-recommends default-jre-headless -q")
    os.system("java -version")

## 3. PyTorch
- **RunPod** (CUDA 11.8): pins `torch==2.1.0+cu118`
- **Local RTX 4070** (CUDA 12.x): installs latest torch with cu124

In [ ]:
if IS_RUNPOD:
    # Pin to CUDA 11.8 — prevents accidental upgrades that break CUDA on RunPod
    %pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 \
        --index-url https://download.pytorch.org/whl/cu118 -q
else:
    # Local — install latest torch with CUDA 12.4 support (RTX 4070)
    %pip install torch torchvision \
        --index-url https://download.pytorch.org/whl/cu124 -q

## 4. Other Python dependencies

In [ ]:
%pip install timm pycocoevalcap pycocotools triton bert-score -q

## 5. mamba-ssm CUDA kernel (VMamba speedup)
- **Linux only** — compiles from source, takes **20-40 min**
- **Windows**: not supported — VMamba will use Triton fallback (still works)
- Skip this cell if you don't want to wait

In [ ]:
if IS_WINDOWS:
    print("mamba-ssm requires Linux + CUDA. Skipping on Windows.")
    print("VMamba will use Triton/PyTorch fallback — training still works.")
else:
    %pip install mamba-ssm --no-build-isolation

## 6. Cache pretrained weights

In [ ]:
import os
if IS_RUNPOD:
    os.environ["HF_HOME"] = "/workspace/hf_cache"
    os.makedirs("/workspace/hf_cache", exist_ok=True)
else:
    cache = os.path.join(PROJECT, ".hf_cache")
    os.environ["HF_HOME"] = cache
    os.makedirs(cache, exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

## 7. Working directory + path

In [ ]:
import sys, os
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("Working directory:", os.getcwd())

## 8. Download dataset

In [ ]:
%run src/data/make_data.py
%run src/data/preprocess_annotations.py

## 9. Verify setup

In [ ]:
import torch, shutil
from src.models.encoder_vmamba import WITH_MAMBA_SSM

print(f"Platform      : {platform.system()}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA kernel   : {WITH_MAMBA_SSM}  <- True = fast VMamba selective scan")
print(f"Java (metrics): {shutil.which('java') is not None}  <- True = METEOR/CIDEr will work")
print(f"Flickr8k data : {os.path.exists('data/flickr8k_annotations.json')}  <- True = dataset ready")

## 9b. (Optional) Set up SPICE metric
Skip this cell if you don't want SPICE scores during validation. Downloads `spice-1.0.jar` (~5 MB) and Stanford CoreNLP models (~377 MB, one-time). Linux/RunPod only — needs Java (already installed in cell 2).

In [ ]:
SPICE_DIR = os.path.join(PROJECT, "src", "models", "spice")
SPICE_JAR = os.path.join(SPICE_DIR, "spice-1.0.jar")
SPICE_LIB = os.path.join(SPICE_DIR, "lib", "stanford-corenlp-3.6.0.jar")

if IS_WINDOWS:
    print("SPICE setup is Linux/RunPod-focused. Skipping on Windows — install Java + jar manually if needed.")
else:
    if not os.path.exists(SPICE_JAR):
        print("Downloading spice-1.0.jar ...")
        os.makedirs(SPICE_DIR, exist_ok=True)
        url = "https://github.com/Aldenhovel/bleu-rouge-meteor-cider-spice-eval4imagecaption/raw/main/spice/spice-1.0.jar"
        subprocess.run(["wget", "-q", "-O", SPICE_JAR, url], check=True)
    print(f"spice-1.0.jar : {os.path.exists(SPICE_JAR)}")

    if not os.path.exists(SPICE_LIB):
        print("Downloading Stanford CoreNLP models (~377 MB, one-time) ...")
        from src.models.spice.get_stanford_models import get_stanford_models
        get_stanford_models()
    print(f"Stanford CoreNLP : {os.path.exists(SPICE_LIB)}")

## 10. Config — edit before training

In [ ]:
# End-to-end fine-tune: encoder + decoder are both trained.
# (Memory-saver: VMamba encoder + transformer decoder gradient checkpointing is on by default in train.py.)

ENCODER    = "vmamba_small"   # vit_base | vit_small | vmamba_small
DECODER    = "transformer"    # transformer | mamba | mamba3
DATASET    = "flickr8k"       # flickr8k | mscoco

# VRAM budget for unfrozen vmamba_small + transformer decoder on a 24 GB GPU:
#   BATCH_SIZE 8, GRAD_ACCUM 8  →  effective 64
# Lower BATCH_SIZE if you OOM, raise GRAD_ACCUM to keep the same effective batch.
BATCH_SIZE = 32 
GRAD_ACCUM = 4

EPOCHS     = 30
LR         = 2e-4               # lower than frozen-encoder runs; full fine-tune is more sensitive
WORKERS    = 4 if IS_WINDOWS else 8

# Early stopping — halt if CIDEr does not improve for N epochs (0 disables)
PATIENCE   = 5
MIN_DELTA  = 0.01

# Optional metrics
USE_SPICE  = True    # True requires cell 9b to have run successfully

# Resume from a previous checkpoint (e.g. after a RunPod restart). Empty = train from scratch.
# Tip: on RunPod, models/ lives under /workspace/<repo>, which persists if /workspace is a network volume.
RESUME     = ""       # e.g. "models/last.pt"

## 11. Train

In [ ]:
import subprocess, sys

# Note: --freeze_encoder is intentionally NOT passed — encoder is fine-tuned end-to-end.
cmd = [
    sys.executable, "-u", "src/models/train.py",
    "--dataset",    DATASET,
    "--encoder",    ENCODER,
    "--decoder",    DECODER,
    "--batch_size", str(BATCH_SIZE),
    "--grad_accum", str(GRAD_ACCUM),
    "--lr",         str(LR),
    "--epochs",     str(EPOCHS),
    "--workers",    str(WORKERS),
    "--patience",   str(PATIENCE),
    "--min_delta",  str(MIN_DELTA),
]
if USE_SPICE:
    cmd.append("--spice")
if RESUME:
    cmd += ["--checkpoint", RESUME]

print("Running:", " ".join(cmd))
print(f"Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print("-" * 80)

# Stream output live so you see per-epoch progress in the notebook (subprocess.run buffers).
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    raise
print("-" * 80)
print(f"Exit code: {proc.returncode}")